<a href="https://colab.research.google.com/github/SaadAh-mad/speaker-verification-experiments/blob/master/Exp9_usingexp5_AAMSoftmax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "macabdul9/VoxCeleb_test"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/356 [00:00<?, ?B/s]

data/test-00000-of-00003.parquet:   0%|          | 0.00/441M [00:00<?, ?B/s]

data/test-00001-of-00003.parquet:   0%|          | 0.00/412M [00:00<?, ?B/s]

data/test-00002-of-00003.parquet:   0%|          | 0.00/435M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4874 [00:00<?, ? examples/s]

In [ ]:
###TRAINING

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
from torch.utils.data import Dataset, DataLoader
import os
import gc
import glob
from transformers import (
    WavLMModel,
    Wav2Vec2FeatureExtractor
)
from datasets import load_dataset

hf_dataset = load_dataset(
    "macabdul9/VoxCeleb_test"
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

#################################################
# ASTP
#################################################

class ASTP(nn.Module):

    def __init__(self, in_dim, bottleneck_dim=128):
        super().__init__()

        self.linear1 = nn.Conv1d(
            in_dim,
            bottleneck_dim,
            kernel_size=1
        )

        self.linear2 = nn.Conv1d(
            bottleneck_dim,
            in_dim,
            kernel_size=1
        )

    def forward(self, x):

        alpha = torch.tanh(
            self.linear1(x)
        )

        alpha = torch.softmax(
            self.linear2(alpha),
            dim=2
        )

        mean = torch.sum(
            alpha * x,
            dim=2
        )

        var = torch.sum(
            alpha * (x ** 2),
            dim=2
        ) - mean ** 2

        std = torch.sqrt(
            var.clamp(min=1e-7)
        )

        return torch.cat(
            [mean, std],
            dim=1
        )


#################################################
# Processor
#################################################

processor = Wav2Vec2FeatureExtractor.from_pretrained(
    "microsoft/wavlm-base"
)

def collate_fn(batch):

    audios = [x[0] for x in batch]

    labels = torch.tensor(
        [x[1] for x in batch]
    )

    features = processor(
        audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    return features.input_values, labels

#################################################
# Dataset
#################################################

class VoxCelebDataset(Dataset):

    def __init__(self, hf_split):

        self.data = hf_split

        speakers = sorted(
            list(
                set(
                    sample["speaker_id"]
                    for sample in hf_split
                )
            )
        )

        self.spk2idx = {
            spk: idx
            for idx, spk in enumerate(speakers)
        }

        self.audio_cache = []
        self.labels = []

        for sample in hf_split:

            audio = (
                sample["audio"]
                .get_all_samples()
                .data
                .squeeze(0)
                .numpy()
            )

            #self.audio_cache.append(audio)

            self.labels.append(
                self.spk2idx[
                    sample["speaker_id"]
                ]
            )

    def __len__(self):
        return len(self.audio_cache)

    def __getitem__(self, idx):

        audio = self.audio_cache[idx]

        if len(audio) > 48000:

            start = np.random.randint(
                0,
                len(audio) - 48000
            )

            audio = audio[
                start:start+48000
            ]

        label = self.labels[idx]

        return audio, label

dataset = VoxCelebDataset(
    hf_dataset["test"]
)

print("Files:", len(dataset))
print("Speakers:", len(dataset.spk2idx))

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

#################################################
# AAM-SoftMax
#################################################

class AAMSoftmax(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_classes,
        margin=0.2,
        scale=30
    ):
        super().__init__()

        self.margin = margin
        self.scale = scale

        self.weight = nn.Parameter(
            torch.randn(
                num_classes,
                embedding_dim
            )
        )

        nn.init.xavier_normal_(
            self.weight
        )

    def forward(
        self,
        embeddings,
        labels
    ):

        embeddings = F.normalize(
            embeddings,
            dim=1
        )

        weight = F.normalize(
            self.weight,
            dim=1
        )

        cosine = F.linear(
            embeddings,
            weight
        )

        theta = torch.acos(
            torch.clamp(
                cosine,
                -1 + 1e-7,
                1 - 1e-7
            )
        )

        target_cosine = torch.cos(
            theta + self.margin
        )

        one_hot = torch.zeros_like(
            cosine
        )

        one_hot.scatter_(
            1,
            labels.view(-1,1),
            1
        )

        logits = (
            one_hot * target_cosine
            + (1 - one_hot) * cosine
        )

        logits *= self.scale

        return logits

#################################################
# Model
#################################################

class WavLMSpeakerModel(nn.Module):

    def __init__(self, num_speakers):

        super().__init__()

        self.wavlm = WavLMModel.from_pretrained(
        "microsoft/wavlm-base"
)
        self.layer_weights = nn.Parameter(
            torch.ones(13)
        )

        for p in self.wavlm.parameters():
          p.requires_grad = False

        self.pooling = ASTP(768)

        self.aam = AAMSoftmax(
        embedding_dim=1536,
        num_classes=num_speakers
)

    def forward(
        self,
        input_values,
        labels=None
):

        with torch.no_grad():
          outputs = self.wavlm(
          input_values=input_values,
          output_hidden_states=True
    )
        hidden_states = outputs.hidden_states

        weights = torch.softmax(
            self.layer_weights,
            dim=0
        )

        combined = 0

        for w, h in zip(weights, hidden_states):
            combined = combined + w * h

        combined = combined.transpose(1, 2)

        embedding = self.pooling(
            combined
        )

        embedding = F.normalize(
            embedding,
            dim=-1
        )

        logits = None

        if labels is not None:

            logits = self.aam(
            embedding,
            labels
    )

        return logits, embedding


#################################################
# Load Model
#################################################

model = WavLMSpeakerModel(
    num_speakers=len(dataset.spk2idx)
).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    [
        model.layer_weights,
        *model.pooling.parameters(),
        *model.aam.parameters()
    ],
    lr=5e-3
)



print("Training started...")


#################################################
# Embedding Extraction
#################################################


best_loss = float("inf")

gc.collect()
torch.cuda.empty_cache()

for epoch in range(20):

    total_loss = 0

    model.train()

    for input_values, labels in loader:

        input_values = input_values.to(
            DEVICE
        )

        labels = labels.to(
            DEVICE
        )

        logits, embeddings = model(
        input_values,
        labels
)

        loss = criterion(
            logits,
            labels
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1} "
        f"Loss = {total_loss:.4f}"
    )



    if total_loss < best_loss:

      best_loss = total_loss

      torch.save(
        model.state_dict(),
        "/content/drive/MyDrive/best_voxceleb_test_wavlm_astp_aam.pth"
    )




print(
    "WavLM ASTP model saved!"
)

print(
    torch.softmax(
        model.layer_weights,
        dim=0
    )
)


Device: cuda


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Files: 4874
Speakers: 40


config.json:   0%|          | 0.00/2.24k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Training started...
Epoch 1 Loss = 3848.5942
Epoch 2 Loss = 2258.1717
Epoch 3 Loss = 1870.1383
Epoch 4 Loss = 1647.5700
Epoch 5 Loss = 1478.5757
Epoch 6 Loss = 1333.3249
Epoch 7 Loss = 1240.3475
Epoch 8 Loss = 1169.8467
Epoch 9 Loss = 1091.5131
Epoch 10 Loss = 1037.8706
Epoch 11 Loss = 972.8792
Epoch 12 Loss = 931.1302
Epoch 13 Loss = 914.2907
Epoch 14 Loss = 875.1310
Epoch 15 Loss = 845.4286
Epoch 16 Loss = 825.8507
Epoch 17 Loss = 794.9979
Epoch 18 Loss = 761.0372
Epoch 19 Loss = 746.7574
Epoch 20 Loss = 737.5577
WavLM ASTP model saved!
tensor([0.2407, 0.0305, 0.0030, 0.0038, 0.2200, 0.2272, 0.1578, 0.0659, 0.0121,
        0.0064, 0.0093, 0.0216, 0.0019], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)


In [ ]:
#####################################
# Evaluation
#####################################

#Evaluation script
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
from torch.utils.data import Dataset, DataLoader
import os
import gc
import glob
from transformers import (
    WavLMModel,
    Wav2Vec2FeatureExtractor
)
import random
import numpy as np
import glob
import os
from datasets import load_dataset

hf_dataset = load_dataset(
    "macabdul9/VoxCeleb_test"
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

#################################################
# ASTP
#################################################

class ASTP(nn.Module):

    def __init__(self, in_dim, bottleneck_dim=128):
        super().__init__()

        self.linear1 = nn.Conv1d(
            in_dim,
            bottleneck_dim,
            kernel_size=1
        )

        self.linear2 = nn.Conv1d(
            bottleneck_dim,
            in_dim,
            kernel_size=1
        )

    def forward(self, x):

        alpha = torch.tanh(
            self.linear1(x)
        )

        alpha = torch.softmax(
            self.linear2(alpha),
            dim=2
        )

        mean = torch.sum(
            alpha * x,
            dim=2
        )

        var = torch.sum(
            alpha * (x ** 2),
            dim=2
        ) - mean ** 2

        std = torch.sqrt(
            var.clamp(min=1e-7)
        )

        return torch.cat(
            [mean, std],
            dim=1
        )


#################################################
# Processor
#################################################

processor = Wav2Vec2FeatureExtractor.from_pretrained(
    "microsoft/wavlm-base"
)

def collate_fn(batch):

    audios = [x[0] for x in batch]

    labels = torch.tensor(
        [x[1] for x in batch]
    )

    features = processor(
        audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    return features.input_values, labels

#################################################
# Dataset
#################################################

class VoxCelebDataset(Dataset):

    def __init__(self, hf_split):

        self.data = hf_split

        speakers = sorted(
            list(
                set(
                    sample["speaker_id"]
                    for sample in hf_split
                )
            )
        )

        self.spk2idx = {
            spk: idx
            for idx, spk in enumerate(speakers)
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        sample = self.data[idx]

        audio = (
            sample["audio"]
            .get_all_samples()
            .data
            .squeeze(0)
            .numpy()
        )

        audio = audio[:48000]

        label = self.spk2idx[
            sample["speaker_id"]
        ]

        return audio, label

dataset = VoxCelebDataset(
    hf_dataset["test"]
)

print("Files:", len(dataset))
print("Speakers:", len(dataset.spk2idx))

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)


#################################################
# AAM-SoftMax
#################################################

class AAMSoftmax(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_classes,
        margin=0.2,
        scale=30
    ):
        super().__init__()

        self.margin = margin
        self.scale = scale

        self.weight = nn.Parameter(
            torch.randn(
                num_classes,
                embedding_dim
            )
        )

        nn.init.xavier_normal_(
            self.weight
        )

    def forward(
        self,
        embeddings,
        labels
    ):

        embeddings = F.normalize(
            embeddings,
            dim=1
        )

        weight = F.normalize(
            self.weight,
            dim=1
        )

        cosine = F.linear(
            embeddings,
            weight
        )

        theta = torch.acos(
            torch.clamp(
                cosine,
                -1 + 1e-7,
                1 - 1e-7
            )
        )

        target_cosine = torch.cos(
            theta + self.margin
        )

        one_hot = torch.zeros_like(
            cosine
        )

        one_hot.scatter_(
            1,
            labels.view(-1,1),
            1
        )

        logits = (
            one_hot * target_cosine
            + (1 - one_hot) * cosine
        )

        logits *= self.scale

        return logits

#################################################
# Model
#################################################

class WavLMSpeakerModel(nn.Module):

    def __init__(self, num_speakers):

        super().__init__()

        self.wavlm = WavLMModel.from_pretrained(
        "microsoft/wavlm-base"
)


        self.layer_weights = nn.Parameter(
            torch.ones(13)
        )

        for p in self.wavlm.parameters():
          p.requires_grad = False

        self.pooling = ASTP(768)

        self.aam = AAMSoftmax(
    embedding_dim=1536,
    num_classes=num_speakers
)
    def forward(self, input_values):

        with torch.no_grad():
          outputs = self.wavlm(
          input_values=input_values,
          output_hidden_states=True
    )
        hidden_states = outputs.hidden_states

        weights = torch.softmax(
            self.layer_weights,
            dim=0
        )

        combined = 0

        for w, h in zip(weights, hidden_states):
            combined = combined + w * h

        combined = combined.transpose(1, 2)

        embedding = self.pooling(
            combined
        )

        embedding = F.normalize(
            embedding,
            dim=-1
        )



        return None, embedding


#################################################
# Load Model
#################################################

model = WavLMSpeakerModel(
    num_speakers=len(dataset.spk2idx)
).to(DEVICE)

model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/best_voxceleb_test_wavlm_astp_aam.pth",
        map_location=DEVICE
    )
)

model.eval()
print("Model loaded!")
print(
    torch.softmax(
        model.layer_weights,
        dim=0
    ).detach().cpu()
)

embeddings_cache = {}

def get_embedding_from_index(idx):
    if idx in embeddings_cache:
        return embeddings_cache[idx]

    sample = hf_dataset["test"][idx]

    audio = (
        sample["audio"]
        .get_all_samples()
        .data
        .squeeze(0)
        .numpy()
    )

    audio = audio[:48000]

    features = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    input_values = features.input_values.to(
        DEVICE
    )

    with torch.no_grad():

        _, embedding = model(
            input_values
        )
    embeddings_cache[idx] = embedding
    return embedding

speaker_files = {}

for idx in range(len(hf_dataset["test"])):

    speaker = hf_dataset["test"][idx]["speaker_id"]

    if speaker not in speaker_files:
        speaker_files[speaker] = []

    speaker_files[speaker].append(idx)





##SAME SPEAKER LOOP
same_scores = []

for _ in range(1000):

    speaker = random.choice(
        list(speaker_files.keys())
    )

    files = speaker_files[speaker]

    if len(files) < 2:
        continue

    idx1, idx2 = random.sample(
    files,
    2
)

    emb1 = get_embedding_from_index(idx1)
    emb2 = get_embedding_from_index(idx2)

    score = F.cosine_similarity(
        emb1,
        emb2
    ).item()

    same_scores.append(score)


##DIFFERENT SPEAKER LOOP
different_scores = []

speakers = list(
    speaker_files.keys()
)

for _ in range(1000):

    spk1, spk2 = random.sample(
        speakers,
        2
    )

    idx1 = random.choice(
    speaker_files[spk1]
)

    idx2 = random.choice(
    speaker_files[spk2]
)

    emb1 = get_embedding_from_index(idx1)
    emb2 = get_embedding_from_index(idx2)

    score = F.cosine_similarity(
        emb1,
        emb2
    ).item()

    different_scores.append(score)



#ACCURACY COMPUTE USING 0.42 as threshold
threshold = 0.28

correct_same = sum(
    s > threshold
    for s in same_scores
)

correct_diff = sum(
    s < threshold
    for s in different_scores
)

accuracy = (
    correct_same +
    correct_diff
) / (
    len(same_scores) +
    len(different_scores)
)

print("\nRESULTS\n")

print(
    f"Average Same Speaker Score: "
    f"{np.mean(same_scores):.4f}"
)

print(
    f"Average Different Speaker Score: "
    f"{np.mean(different_scores):.4f}"
)

print(
    f"Gap: "
    f"{np.mean(same_scores)-np.mean(different_scores):.4f}"
)

print(
    f"Accuracy: {accuracy:.4f}"
)

print("\nSame Speaker Statistics")
print("Min:", np.min(same_scores))
print("Max:", np.max(same_scores))
print("Std:", np.std(same_scores))

print("\nDifferent Speaker Statistics")
print("Min:", np.min(different_scores))
print("Max:", np.max(different_scores))
print("Std:", np.std(different_scores))


Device: cuda
Files: 4874
Speakers: 40


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Model loaded!
tensor([0.2407, 0.0305, 0.0030, 0.0038, 0.2200, 0.2272, 0.1578, 0.0659, 0.0121,
        0.0064, 0.0093, 0.0216, 0.0019])

RESULTS

Average Same Speaker Score: 0.3898
Average Different Speaker Score: 0.1671
Gap: 0.2227
Accuracy: 0.6840

Same Speaker Statistics
Min: 0.052823781967163086
Max: 0.6464139223098755
Std: 0.09271854620491163

Different Speaker Statistics
Min: -0.10057250410318375
Max: 0.46359783411026
Std: 0.09405430011653233


In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

labels = (
    [1] * len(same_scores)
    +
    [0] * len(different_scores)
)

scores = same_scores + different_scores

fpr, tpr, thresholds = roc_curve(
    labels,
    scores
)

fnr = 1 - tpr

eer_idx = np.nanargmin(
    np.abs(fnr - fpr)
)

eer = (
    fpr[eer_idx]
    +
    fnr[eer_idx]
) / 2

print(f"EER: {eer*100:.2f}%")
print(
    f"EER Threshold: "
    f"{thresholds[eer_idx]:.4f}"
)

EER: 11.70%
EER Threshold: 0.2754


In [ ]:
best_acc = 0
best_threshold = 0

for threshold in np.arange(-0.1, 0.8, 0.01):

    correct_same = sum(
        s > threshold
        for s in same_scores
    )

    correct_diff = sum(
        s < threshold
        for s in different_scores
    )

    acc = (
        correct_same +
        correct_diff
    ) / (
        len(same_scores) +
        len(different_scores)
    )

    if acc > best_acc:
        best_acc = acc
        best_threshold = threshold

print(best_threshold)
print(best_acc)

0.2699999999999998
0.887
